# qubox_v2 Hardware Acceptance Notebook

**Purpose:** Step-by-step hardware acceptance testing for the qubox_v2 declarative architecture.

**Date:** _____________  
**Operator:** _____________  
**Build Hash:** (populated automatically below)

---

## Prerequisites

- [ ] Offline audit passed (`python audit_offline.py`)
- [ ] OPX hardware connected and powered
- [ ] Qubit at known operating point

## Phase 1: Session Setup & Connectivity

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

CONFIG_DIR = Path("seq_1_device/config")
print(f"Config directory: {CONFIG_DIR}")
print(f"Exists: {CONFIG_DIR.exists()}")

In [ ]:
# Build SessionState
from qubox_v2.core.session_state import SessionState

ss = SessionState.from_config_dir(CONFIG_DIR)
print(ss.summary())
print(f"\nBuild hash: {ss.build_hash}")

In [ ]:
# Schema validation
from qubox_v2.core.schemas import validate_config_dir

results = validate_config_dir(CONFIG_DIR)
for r in results:
    status = "PASS" if r.valid else "FAIL"
    print(f"  {status} v={r.version}")
    for e in r.errors:
        print(f"    ERROR: {e}")
    for w in r.warnings:
        print(f"    WARN: {w}")

In [ ]:
# Connect to QMM
from qm.QuantumMachinesManager import QuantumMachinesManager

qmm = QuantumMachinesManager()
print(f"Open QMs: {qmm.list_open_quantum_machines()}")

qm = qmm.open_qm(ss.hardware)
print(f"QM opened: {qm.id}")

## Phase 2: Constant Pulse Smoke Test

In [ ]:
from qm.qua import *
import time

# Play constant pulse on each element
elements = list(ss.hardware.get("elements", {}).keys())
print(f"Elements: {elements}")

for element in elements:
    print(f"\nTesting: {element}")
    try:
        with program() as const_prog:
            with infinite_loop_():
                play("const", element)
                wait(1000, element)
        
        job = qm.execute(const_prog)
        time.sleep(0.5)
        job.halt()
        
        report = job.execution_report()
        if report.has_errors():
            print(f"  FAIL: {report.errors()}")
        else:
            print(f"  PASS")
    except Exception as exc:
        print(f"  SKIP/FAIL: {exc}")

## Phase 3: PulseFactory Waveform Verification

In [ ]:
# Compile all pulse specs
from qubox_v2.pulses.factory import PulseFactory

if ss.pulse_specs.get("specs"):
    factory = PulseFactory(ss.pulse_specs)
    compiled = factory.compile_all()
    print(f"Compiled {len(compiled)} pulse specs")
    
    for name, (I_wf, Q_wf, meta) in compiled.items():
        print(f"  {name}: shape={meta.get('shape')}, length={len(I_wf)}")
else:
    print("No pulse_specs found (using legacy pulses.json)")
    compiled = {}

In [ ]:
# Plot key waveforms
key_pulses = ["ref_r180", "x180", "x90", "y180", "y90"]
available = [p for p in key_pulses if p in compiled]

if available:
    fig, axes = plt.subplots(len(available), 1, figsize=(10, 3 * len(available)))
    if len(available) == 1:
        axes = [axes]
    
    for ax, name in zip(axes, available):
        I_wf, Q_wf, meta = compiled[name]
        t = np.arange(len(I_wf))
        ax.plot(t, I_wf, label="I", color="blue")
        ax.plot(t, Q_wf, label="Q", color="red")
        ax.set_title(f"{name} (shape={meta.get('shape')})")
        ax.set_xlabel("Sample")
        ax.set_ylabel("Amplitude")
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No key pulses available to plot")

## Phase 4: CalibrationStateMachine Lifecycle Test

In [ ]:
from qubox_v2.calibration.state_machine import (
    CalibrationStateMachine, CalibrationState,
    CalibrationPatch, PatchValidation,
)

sm = CalibrationStateMachine(experiment="hw_acceptance_rabi")
print(f"Initial state: {sm.state.value}")

# Walk through lifecycle
for target in [
    CalibrationState.CONFIGURED,
    CalibrationState.ACQUIRING,
]:
    sm.transition(target)
    print(f"  -> {sm.state.value}")

print(f"\nReady for acquisition. Run Rabi experiment in next cell.")

In [ ]:
# Run minimal Power Rabi
n_avg = 200
amplitudes = np.linspace(0.0, 0.45, 50)

with program() as rabi_prog:
    n = declare(int)
    a = declare(fixed)
    I = declare(fixed)
    Q = declare(fixed)
    I_stream = declare_stream()
    Q_stream = declare_stream()

    with for_(n, 0, n < n_avg, n + 1):
        with for_each_(a, amplitudes.tolist()):
            play("x180" * amp(a), "qubit")
            align("qubit", "resonator")
            measure(
                "readout", "resonator", None,
                dual_demod.full("cos", "out1", "sin", "out2", I),
                dual_demod.full("minus_sin", "out1", "cos", "out2", Q),
            )
            save(I, I_stream)
            save(Q, Q_stream)

    with stream_processing():
        I_stream.buffer(len(amplitudes)).average().save("I")
        Q_stream.buffer(len(amplitudes)).average().save("Q")

job = qm.execute(rabi_prog)
result_handles = job.result_handles
result_handles.wait_for_all_values()

I_data = result_handles.get("I").fetch_all()
Q_data = result_handles.get("Q").fetch_all()

print(f"Acquired {len(I_data)} points")

# Transition to ACQUIRED
sm.transition(CalibrationState.ACQUIRED)
print(f"State: {sm.state.value}")

In [ ]:
# Plot Rabi oscillation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(amplitudes, I_data, 'o-', markersize=3)
ax1.set_xlabel("Drive amplitude")
ax1.set_ylabel("I (V)")
ax1.set_title("Power Rabi - I channel")
ax1.grid(True, alpha=0.3)

ax2.plot(amplitudes, Q_data, 'o-', markersize=3, color='red')
ax2.set_xlabel("Drive amplitude")
ax2.set_ylabel("Q (V)")
ax2.set_title("Power Rabi - Q channel")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze: fit sinusoidal to find pi-pulse amplitude
sm.transition(CalibrationState.ANALYZING)
print(f"State: {sm.state.value}")

from scipy.optimize import curve_fit

def rabi_model(x, A, f, phi, offset):
    return A * np.cos(2 * np.pi * f * x + phi) + offset

try:
    signal = np.sqrt(I_data**2 + Q_data**2)
    p0 = [np.ptp(signal)/2, 2.0, 0, np.mean(signal)]
    popt, pcov = curve_fit(rabi_model, amplitudes, signal, p0=p0)
    
    pi_amp = 1 / (2 * abs(popt[1]))
    
    # Goodness of fit
    residuals = signal - rabi_model(amplitudes, *popt)
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((signal - np.mean(signal))**2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    print(f"Fitted pi-pulse amplitude: {pi_amp:.5f}")
    print(f"R-squared: {r_squared:.4f}")
    
    # Create patch
    patch = CalibrationPatch(experiment="hw_acceptance_rabi")
    
    # Get current amplitude from calibration
    old_amp = 0.11165  # Replace with actual current value
    patch.add_change(
        path="pulse_calibrations.qubit.x180.amplitude",
        old_value=old_amp,
        new_value=float(pi_amp),
    )
    patch.validation = PatchValidation(
        passed=r_squared > 0.95,
        checks={"min_r2": r_squared > 0.95, "bounds_check": 0.01 < pi_amp < 0.45},
    )
    
    sm.patch = patch
    print(f"\nPatch attached. Validation passed: {patch.validation.passed}")
    print(patch.summary())
    
except Exception as exc:
    print(f"Fit failed: {exc}")
    pi_amp = None
    r_squared = 0

In [ ]:
# Complete state machine lifecycle
sm.transition(CalibrationState.ANALYZED)
sm.transition(CalibrationState.PENDING_APPROVAL)

print(f"State: {sm.state.value}")
print(f"Committable: {sm.is_committable()}")
print(f"\nState machine summary:")
summary = sm.summary()
for k, v in summary.items():
    print(f"  {k}: {v}")

## Phase 5: GE Discrimination

In [ ]:
# Run GE discrimination
n_shots = 1000

with program() as ge_prog:
    n = declare(int)
    I_g = declare(fixed)
    Q_g = declare(fixed)
    I_e = declare(fixed)
    Q_e = declare(fixed)
    I_g_st = declare_stream()
    Q_g_st = declare_stream()
    I_e_st = declare_stream()
    Q_e_st = declare_stream()

    with for_(n, 0, n < n_shots, n + 1):
        # Ground
        align("qubit", "resonator")
        measure("readout", "resonator", None,
                dual_demod.full("cos", "out1", "sin", "out2", I_g),
                dual_demod.full("minus_sin", "out1", "cos", "out2", Q_g))
        save(I_g, I_g_st)
        save(Q_g, Q_g_st)
        wait(250000 // 4, "resonator")

        # Excited
        play("x180", "qubit")
        align("qubit", "resonator")
        measure("readout", "resonator", None,
                dual_demod.full("cos", "out1", "sin", "out2", I_e),
                dual_demod.full("minus_sin", "out1", "cos", "out2", Q_e))
        save(I_e, I_e_st)
        save(Q_e, Q_e_st)
        wait(250000 // 4, "resonator")

    with stream_processing():
        I_g_st.save_all("I_g")
        Q_g_st.save_all("Q_g")
        I_e_st.save_all("I_e")
        Q_e_st.save_all("Q_e")

job = qm.execute(ge_prog)
result_handles = job.result_handles
result_handles.wait_for_all_values()

I_g_data = result_handles.get("I_g").fetch_all()
Q_g_data = result_handles.get("Q_g").fetch_all()
I_e_data = result_handles.get("I_e").fetch_all()
Q_e_data = result_handles.get("Q_e").fetch_all()

print(f"Acquired {len(I_g_data)} ground + {len(I_e_data)} excited shots")

In [ ]:
# Compute discrimination metrics
g_complex = I_g_data + 1j * Q_g_data
e_complex = I_e_data + 1j * Q_e_data
g_mean = np.mean(g_complex)
e_mean = np.mean(e_complex)

separation = abs(e_mean - g_mean)
g_std = np.std(np.abs(g_complex - g_mean))
e_std = np.std(np.abs(e_complex - e_mean))
snr = separation / ((g_std + e_std) / 2) if (g_std + e_std) > 0 else 0

# Threshold fidelity
angle = np.angle(e_mean - g_mean)
g_rot = np.real(g_complex * np.exp(-1j * angle))
e_rot = np.real(e_complex * np.exp(-1j * angle))
threshold = (np.mean(g_rot) + np.mean(e_rot)) / 2
fidelity = (np.sum(g_rot < threshold) + np.sum(e_rot >= threshold)) / (2 * n_shots)

print(f"Separation: {separation:.6f}")
print(f"SNR: {snr:.2f}")
print(f"Fidelity: {fidelity:.2%}")
print(f"\nAcceptance: SNR > 1.0: {'PASS' if snr > 1 else 'FAIL'}")
print(f"Acceptance: Fidelity > 90%: {'PASS' if fidelity > 0.9 else 'FAIL'}")

In [ ]:
# Plot IQ scatter
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(I_g_data * 1e3, Q_g_data * 1e3, alpha=0.3, s=5, label="Ground", color="blue")
ax.scatter(I_e_data * 1e3, Q_e_data * 1e3, alpha=0.3, s=5, label="Excited", color="red")
ax.scatter(np.real(g_mean) * 1e3, np.imag(g_mean) * 1e3, s=100, marker="x",
           color="darkblue", linewidths=2, label="G mean")
ax.scatter(np.real(e_mean) * 1e3, np.imag(e_mean) * 1e3, s=100, marker="x",
           color="darkred", linewidths=2, label="E mean")
ax.set_xlabel("I (mV)")
ax.set_ylabel("Q (mV)")
ax.set_title(f"GE Discrimination (Fidelity={fidelity:.1%}, SNR={snr:.1f})")
ax.legend()
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Phase 6: Artifact Verification

In [ ]:
from qubox_v2.core.artifact_manager import ArtifactManager

am = ArtifactManager(CONFIG_DIR.parent, ss.build_hash)
print(f"Artifact root: {am.root}")

# Save session state
state_path = am.save_session_state(ss.to_dict())
print(f"Session state: {state_path}")

# Save acceptance report
report_content = f"""# Hardware Acceptance Report
- Build hash: {ss.build_hash}
- Rabi pi-amp: {pi_amp if pi_amp else 'N/A'}
- GE SNR: {snr:.2f}
- GE Fidelity: {fidelity:.2%}
"""
report_path = am.save_report("hardware_acceptance", report_content)
print(f"Report: {report_path}")

# List all artifacts
print(f"\nAll artifacts:")
for a in am.list_artifacts():
    print(f"  {a.name}")

## Summary & Sign-Off

In [ ]:
print("=" * 50)
print("  HARDWARE ACCEPTANCE SUMMARY")
print("=" * 50)
print(f"  Build hash:  {ss.build_hash}")
print(f"  Const pulse: {'PASS' if True else 'FAIL'}")
print(f"  Rabi fit:    {'PASS' if pi_amp and r_squared > 0.95 else 'FAIL'}")
print(f"  GE SNR:      {snr:.2f} {'PASS' if snr > 1 else 'FAIL'}")
print(f"  GE Fidelity: {fidelity:.1%} {'PASS' if fidelity > 0.9 else 'FAIL'}")
print(f"  Artifacts:   saved to {am.root}")
print("=" * 50)

all_pass = (pi_amp is not None and r_squared > 0.95 and snr > 1 and fidelity > 0.9)
print(f"\n  OVERALL: {'ACCEPTED' if all_pass else 'NOT ACCEPTED'}")

In [ ]:
# Clean up
try:
    qm.close()
    print("QM closed.")
except Exception:
    pass